In [ ]:
# 1. Clonar el repositorio
!git clone https://github.com/felipe-astudillo-s/CPMP-Transformer.git

# 2. Entrar a la carpeta del proyecto
%cd CPMP-Transformer

In [ ]:
%cd CPMP-Framework

In [ ]:
# Compilamos solo los archivos necesarios, ignorando test.cpp
!g++ Codigo_C_solver/Greedy.cpp Codigo_C_solver/Layout.cpp Codigo_C_solver/Bsg.cpp Codigo_C_solver/main_cpmp.cpp -o Codigo_C_solver/frg -O3 -std=c++11

# Le damos permisos de ejecución al nuevo archivo generado
!chmod +x Codigo_C_solver/frg

In [ ]:
import sys
import os

# Agregamos la carpeta src al path
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

In [ ]:
# Montar Google Drive (se pedirá autorización una sola vez)
from google.colab import drive
drive.mount('/content/drive')

# Carpeta destino en Drive — se crea si no existe
DRIVE_DEST = '/content/drive/MyDrive/CPMP_Data/V10'
os.makedirs(DRIVE_DEST, exist_ok=True)
print(f'📂 Carpeta Drive lista: {DRIVE_DEST}')

In [ ]:
import shutil
from generation.instances import generate_instances
from generation.data import generate_data
from generation.adapters import TacticalStackMatrixAdapter, DefaultMovesAdapter

# Mismas configuraciones que V9 victorioso
configuraciones = [
    # (H, S, N, amount)
    (5, 4, 15, 50000),   # Tableros pequeños (fáciles)
    (7, 5, 25, 100000),  # Benchmark actual (intermedios)
    (10, 6, 45, 50000),  # Tableros grandes y altos (difíciles)
    (6, 7, 30, 50000)    # Tableros anchos pero bajos
]

r_desorden = 50
max_steps = 50
seed = 42

print('🚀 Iniciando pipeline de generación V10 (TacticalStackMatrixAdapter)...\n')

for H, S, N, amount in configuraciones:
    filename = f'E{S}-{N}-H{H}_V10'
    print(f'--- Procesando Configuración: {S} Stacks, {N} Contenedores, Altura {H} ---')

    # 1. Generar instancias brutas
    print(f'Generando {amount} instancias brutas...')
    generate_instances(filename, H, S, N, amount, r_desorden, seed)

    # 2. Extraer características tácticas y resolver
    print('Resolviendo y empaquetando datos tácticos (5D)...')
    layout_adapter = TacticalStackMatrixAdapter()
    moves_adapter = DefaultMovesAdapter()

    generate_data(filename, H, max_steps, layout_adapter, moves_adapter)
    print(f'✅ Finalizado: {filename}.data guardado en data/')

    # 3. Subir automáticamente a Google Drive
    local_path = os.path.join('data', f'{filename}.data')
    drive_path = os.path.join(DRIVE_DEST, f'{filename}.data')
    shutil.copy(local_path, drive_path)
    print(f'☁️  Subido a Drive: {drive_path}\n')

print('🎉 ¡Toda la generación V10 ha terminado!')